### 라이브러리

In [30]:
import pandas as pd
import numpy as np

### 데이터 호출

In [31]:
df = pd.read_csv('train.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 908 entries, 0 to 907
Columns: 2881 entries, PRODUCT_ID to X_2875
dtypes: float64(2876), int64(1), object(4)
memory usage: 20.0+ MB


### STEP1 제품별 분리 

In [32]:
def split_by_product(df, product_col='PRODUCT_CODE'):
    df_A = df[df[product_col] == 'A_31'].copy()
    df_TO = df[df[product_col].isin(["T_31","O_31"])].copy()

    return df_A, df_TO 

### STEP2 누수 컬럼 제거

타깃(`Y_Quality`)과 타깃 파생(`Y_Class`), 식별자(`PRODUCT_ID`)를 피처에서 제외한다. `PRODUCT_CODE`·`TIMESTAMP`와 함께 검증용으로 따로 보관하며, `TIMESTAMP`는 시간 기반 검증을 위해 datetime으로 파싱한다.

In [33]:
def drop_leakage(df,
                 target_col='Y_Quality',
                 leak_cols=['Y_Class'],                 # 타깃 파생 = 누수
                 id_cols=['PRODUCT_ID', 'PRODUCT_CODE'],
                 time_col='TIMESTAMP'):
    y = df[target_col].copy()
    keep = [c for c in leak_cols + id_cols if c in df.columns]
    temp = df[keep].copy()
    if time_col in df.columns:
        temp[time_col] = pd.to_datetime(df[time_col], errors='coerce')   # 시간 기반 검증용 보관
    X = df.drop(columns=[target_col] + keep + ([time_col] if time_col in df.columns else []))
    return X, y, temp

### STEP3 위장 결측 -> NaN

In [34]:
def missing_value(X, placeholder = [999,9999,-999,-9999]):
    X = X.replace(placeholder, np.nan)
    return X

### STEP4 완전 결측 컬럼 제거

In [35]:
def remove_all_nan(X):
    all_nan_cols = X.columns[X.isna().all()].tolist()
    X = X.drop(columns=all_nan_cols)
    return X, all_nan_cols

### STEP5 결측 비율 60% 초과 컬럼 제거

In [36]:
def remove_high_missing(X, threshold=0.6):
    missing_ratio = X.isna().mean()
    drop_cols = missing_ratio[missing_ratio > threshold].index.tolist()
    X = X.drop(columns=drop_cols)
    return X, drop_cols

### STEP6 상수/제로분산 제거

In [37]:
def remove_constant(X):
    nunique = X.columns[X.nunique(dropna=False) <= 1]
    std_zero = X.select_dtypes(include="number").columns[
        X.select_dtypes(include="number").std() == 0
    ]
    
    drop_cols = list(set(nunique) | set(std_zero))
    X = X.drop(columns=drop_cols)
    
    return X, drop_cols

### STEP7 저분산 제거

In [38]:
def remove_low_variance(X, threshold=0.01):
    num_cols = X.select_dtypes(include="number").columns
    if len(num_cols) == 0:
        return X, []

    X_num = X[num_cols]
    col_range = (X_num.max() - X_num.min()).replace(0, np.nan)  # 상수 컬럼은 STEP4에서 이미 제거됨

    X_scaled = (X_num - X_num.min()) / col_range
    low_var_cols = X_scaled.columns[X_scaled.var(skipna=True) < threshold].tolist()

    X = X.drop(columns=low_var_cols)
    return X, low_var_cols

### STEP8 상관계수 높은 컬럼 제거

임계값(0.95)을 넘는 상관 쌍에서 결측이 적은 컬럼을 남긴다(동률이면 앞선 컬럼 유지). 컬럼 순서에 따라 결과가 달라지지 않도록 제거 기준을 고정했다.

In [39]:
def remove_high_correlation(X, threshold=0.95):
    num_cols = X.select_dtypes(include="number").columns
    if len(num_cols) < 2:
        return X, []
    corr = X[num_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    miss = X[num_cols].isna().mean()                     # 결측 적은 쪽을 남긴다
    order = {c: i for i, c in enumerate(num_cols)}
    drop = set()
    for b in upper.columns:
        for a in upper.index:
            v = upper.loc[a, b]
            if pd.notna(v) and v > threshold and a not in drop and b not in drop:
                drop.add(a if (miss[a] > miss[b] or (miss[a] == miss[b] and order[a] > order[b])) else b)
    return X.drop(columns=list(drop)), sorted(drop)

### STEP9 라인 인코딩

`LINE`을 정수형(0/1) 더미로 변환한다. CSV로 저장·재로딩해도 형이 유지된다.

In [40]:
def encode_line(X, line_col='LINE'):
    if line_col in X.columns:
        X = pd.get_dummies(X, columns=[line_col], drop_first=True, dtype=int)
    return X

### 전체 전처리 함수 

In [41]:
def preprocess(
    df,
    target_col='Y_Quality',
    leak_cols=['Y_Class'],
    id_cols=['PRODUCT_ID', 'PRODUCT_CODE'],
    time_col='TIMESTAMP',
    placeholder=[999, 9999, -999, -9999],
    line_col='LINE',
    missing_threshold=0.6,
    variance_threshold=0.01,
    corr_threshold=0.95,
    remove_nan=True
):
    # STEP2 누수 컬럼 제거 (타깃·타깃파생·식별자·시간)
    X, y, temp = drop_leakage(df, target_col, leak_cols, id_cols, time_col)

    # STEP3 위장결측 → NaN
    X = missing_value(X, placeholder)

    # STEP4 완전 결측 컬럼 제거
    if remove_nan:
        X, all_nan_cols = remove_all_nan(X)
    else:
        all_nan_cols = X.columns[X.isna().all()].tolist()

    # STEP5~8
    X, high_missing_cols = remove_high_missing(X, missing_threshold)
    X, constant_cols = remove_constant(X)
    X, low_var_cols = remove_low_variance(X, variance_threshold)
    X, high_corr_cols = remove_high_correlation(X, corr_threshold)

    # STEP9 LINE 인코딩
    X = encode_line(X, line_col)

    print("=" * 50)
    print(f"최종 데이터 크기 : {X.shape}")
    print(f"완전결측 제거 : {len(all_nan_cols)}개")
    print(f"결측비율 {missing_threshold*100:.0f}% 초과 제거 : {len(high_missing_cols)}개")
    print(f"상수/제로분산 제거 : {len(constant_cols)}개")
    print(f"저분산 제거 : {len(low_var_cols)}개")
    print(f"고상관 제거 : {len(high_corr_cols)}개")
    print("=" * 50)
    return X, y, temp

### 전처리 적용

In [42]:
df_A, df_TO = split_by_product(df)
X_A, y_A, temp_A = preprocess(df_A)
X_TO, y_TO, temp_TO = preprocess(df_TO)

print(X_A.shape)

print(X_TO.shape)

최종 데이터 크기 : (316, 574)
완전결측 제거 : 683개
결측비율 60% 초과 제거 : 612개
상수/제로분산 제거 : 225개
저분산 제거 : 43개
고상관 제거 : 741개
최종 데이터 크기 : (592, 339)
완전결측 제거 : 2195개
결측비율 60% 초과 제거 : 14개
상수/제로분산 제거 : 112개
저분산 제거 : 45개
고상관 제거 : 171개
(316, 574)
(592, 339)


### CSV 형식으로 저장

In [43]:
df_A_processed = pd.concat(
    [X_A.reset_index(drop=True), y_A.reset_index(drop=True), temp_A.reset_index(drop=True)], axis=1)
df_A_processed.to_csv("A_31_preprocessed.csv", index=False, encoding="utf-8-sig")

df_TO_processed = pd.concat(
    [X_TO.reset_index(drop=True), y_TO.reset_index(drop=True), temp_TO.reset_index(drop=True)], axis=1)
df_TO_processed.to_csv("TO_31_preprocessed.csv", index=False, encoding="utf-8-sig")

### 수정 내역
- STEP2 drop_leakage — Y_Class(타깃 파생 누수)·PRODUCT_ID를 피처에서 제외, TIMESTAMP는 datetime으로 파싱해 보관
- STEP8 remove_high_correlation — 상관 쌍에서 결측 적은 쪽을 남기는 고정 규칙(순서 의존성 제거)
- STEP9 encode_line — dtype=int 추가
- 저장 셀 — reset_index로 빈(NaN) 행 방지
- preprocess 래퍼는 위 시그니처에 맞게만 조정